In [ ]:
# !pip install yfinance ta-lib vectorbt scipy matplotlib pandas numpy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# FTMO TRADEABLE INSTRUMENT UNIVERSE
FTMO_UNIVERSE = {
    # Indices
    '^DJI': {'name': 'US30', 'class': 'index', 'best_strategy': 'MACD'},
    '^IXIC': {'name': 'US100', 'class': 'index', 'best_strategy': 'MACD'},
    '^GSPC': {'name': 'US500', 'class': 'index', 'best_strategy': 'MACD'},
    '^GDAXI': {'name': 'DAX', 'class': 'index', 'best_strategy': 'MACD'},
    '^N225': {'name': 'Nikkei', 'class': 'index', 'best_strategy': 'MACD'},
    '^FTSE': {'name': 'FTSE', 'class': 'index', 'best_strategy': 'RSI'},
    # Forex
    'EURUSD=X': {'name': 'EURUSD', 'class': 'forex', 'best_strategy': 'RSI'},
    'GBPUSD=X': {'name': 'GBPUSD', 'class': 'forex', 'best_strategy': 'RSI'},
    'USDJPY=X': {'name': 'USDJPY', 'class': 'forex', 'best_strategy': 'MACD'},
    'AUDUSD=X': {'name': 'AUDUSD', 'class': 'forex', 'best_strategy': 'RSI'},
    'USDCAD=X': {'name': 'USDCAD', 'class': 'forex', 'best_strategy': 'RSI'},
    'USDCHF=X': {'name': 'USDCHF', 'class': 'forex', 'best_strategy': 'RSI'},
    'NZDUSD=X': {'name': 'NZDUSD', 'class': 'forex', 'best_strategy': 'RSI'},
    # Crypto
    'BTC-USD': {'name': 'BTCUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'ETH-USD': {'name': 'ETHUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'SOL-USD': {'name': 'SOLUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'LTC-USD': {'name': 'LTCUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'XRP-USD': {'name': 'XRPUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'ADA-USD': {'name': 'ADAUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'DOGE-USD': {'name': 'DOGEUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'LINK-USD': {'name': 'LINKUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    'AVAX-USD': {'name': 'AVAXUSD', 'class': 'crypto', 'best_strategy': 'Donchian'},
    # Commodities
    'GC=F': {'name': 'XAUUSD', 'class': 'commodity', 'best_strategy': 'Donchian'},
    'SI=F': {'name': 'XAGUSD', 'class': 'commodity', 'best_strategy': 'Donchian'},
    'CL=F': {'name': 'Crude Oil', 'class': 'commodity', 'best_strategy': 'Donchian'},
    'NG=F': {'name': 'NatGas', 'class': 'commodity', 'best_strategy': 'Donchian'},
}

# Configuration
LOOKBACK_MONTHS = 12      # Momentum lookback (standard 12-1)
SKIP_LAST_MONTH = True    # Skip most recent month (12-1 momentum)
REBALANCE_FREQ = 'Q'      # Quarterly rebalance
TOP_N = 8                 # How many instruments to select
START_DATE = '2019-01-01'
TREND_SMA = 200           # SMA trend overlay period

print(f"FTMO Universe: {len(FTMO_UNIVERSE)} instruments")
for asset_class in ['index', 'forex', 'crypto', 'commodity']:
    instruments = [v['name'] for v in FTMO_UNIVERSE.values() if v['class'] == asset_class]
    print(f"  {asset_class:12s}: {', '.join(instruments)}")

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# DOWNLOAD ALL INSTRUMENT DATA

print("Downloading data for all FTMO instruments...")
all_data = {}
failed = []

_tickers = list(FTMO_UNIVERSE.keys())
_raw = download_multi(_tickers, START_DATE)

for ticker, info in FTMO_UNIVERSE.items():
    if ticker in _raw and len(_raw[ticker]) > 252:
        close = _raw[ticker]['Close'].astype(float).squeeze()
        all_data[ticker] = close
        print(f"  OK: {info['name']:12s} ({ticker:12s}) — {len(close)} bars")
    else:
        failed.append(ticker)
        print(f"  SKIP: {info['name']} — insufficient data")

print(f"\nLoaded {len(all_data)}/{len(FTMO_UNIVERSE)} instruments")
if failed:
    print(f"Failed/skipped: {[FTMO_UNIVERSE[t]['name'] for t in failed]}")

In [ ]:
# COMPUTE MOMENTUM + QUALITY SCORES (Alpha Architect QM Method)
# All backward-looking — no lookahead

def compute_momentum_scores(prices_dict, as_of_date, lookback_months=12, skip_last=True):
    """
    Compute 12-1 momentum and Frog-in-the-Pan quality for each instrument.
    Uses only data BEFORE as_of_date.
    """
    scores = []
    for ticker, close in prices_dict.items():
        # Only use data up to as_of_date
        close_before = close[close.index < as_of_date]
        if len(close_before) < 252:
            continue
        
        # 12-1 momentum: total return over past 12 months, excluding last month
        end_idx = len(close_before)
        one_month_ago = close_before.iloc[-21:]  # ~1 month
        twelve_months = close_before.iloc[-252:]  # ~12 months
        
        if skip_last:
            # 12-month return minus last month (the classic 12-1)
            mom_12m = (close_before.iloc[-21] / close_before.iloc[-252]) - 1
        else:
            mom_12m = (close_before.iloc[-1] / close_before.iloc[-252]) - 1
        
        # Frog-in-the-Pan: % of positive return days over lookback
        # Higher = smoother, more continuous momentum = BETTER quality
        daily_rets = twelve_months.pct_change().dropna()
        if skip_last:
            daily_rets = daily_rets.iloc[:-21]  # exclude last month
        
        pct_positive = (daily_rets > 0).sum() / len(daily_rets) if len(daily_rets) > 0 else 0
        
        # Momentum quality: consistency of direction
        # Additional quality metric: information discreteness
        # Positive days minus negative days, divided by total days
        pos_days = (daily_rets > 0).sum()
        neg_days = (daily_rets < 0).sum()
        total_days = len(daily_rets)
        frog_score = (pos_days - neg_days) / total_days if total_days > 0 else 0
        
        # If momentum is negative, flip the frog score
        # (for negative momentum, MORE negative days = higher quality bearish momentum)
        if mom_12m < 0:
            frog_score = -frog_score
        
        # Trend filter: is price above SMA(200)?
        sma = close_before.rolling(TREND_SMA).mean()
        above_trend = close_before.iloc[-1] > sma.iloc[-1] if not np.isnan(sma.iloc[-1]) else False
        
        # Realized vol (annualized, last 21 days)
        recent_vol = daily_rets.iloc[-21:].std() * np.sqrt(252) if len(daily_rets) >= 21 else np.nan
        
        scores.append({
            'ticker': ticker,
            'name': FTMO_UNIVERSE[ticker]['name'],
            'asset_class': FTMO_UNIVERSE[ticker]['class'],
            'best_strategy': FTMO_UNIVERSE[ticker]['best_strategy'],
            'momentum_12_1': mom_12m,
            'frog_score': frog_score,
            'pct_positive_days': pct_positive,
            'above_trend': above_trend,
            'realized_vol': recent_vol,
            'last_price': close_before.iloc[-1],
            'price_12m_ago': close_before.iloc[-252],
        })
    
    df = pd.DataFrame(scores)
    
    # Rank: composite score = 0.5 * momentum_rank + 0.5 * quality_rank
    if len(df) > 0:
        df['mom_rank'] = df['momentum_12_1'].rank(ascending=False, pct=True)
        df['quality_rank'] = df['frog_score'].rank(ascending=False, pct=True)
        df['composite_score'] = 0.5 * df['mom_rank'] + 0.5 * df['quality_rank']
        df = df.sort_values('composite_score', ascending=True)  # lower rank = better
    
    return df

# Compute current scores
today = pd.Timestamp.now()
current_scores = compute_momentum_scores(all_data, today)

print("=" * 90)
print("QUANTITATIVE MOMENTUM SCORES — CURRENT RANKINGS")
print("=" * 90)
print(f"As of: {today.date()}")
print(f"Method: 12-1 Momentum + Frog-in-the-Pan Quality (Alpha Architect)")
print(f"Trend Filter: SMA({TREND_SMA})")
print("=" * 90)

display_cols = ['name', 'asset_class', 'best_strategy', 'momentum_12_1', 'frog_score', 
                'pct_positive_days', 'above_trend', 'realized_vol', 'composite_score']
print(current_scores[display_cols].to_string(index=False, float_format='%.3f'))

In [ ]:
# SELECT TOP INSTRUMENTS FOR FTMO TRADING

# Filter: only instruments above their trend (SMA 200)
trending = current_scores[current_scores['above_trend'] == True].copy()
print(f"\nInstruments above SMA({TREND_SMA}): {len(trending)} / {len(current_scores)}")

# Take top N by composite score
selected = trending.head(TOP_N) if len(trending) >= TOP_N else trending
if len(selected) < TOP_N:
    # Fill remaining slots from below-trend instruments with best momentum
    remaining = current_scores[~current_scores['ticker'].isin(selected['ticker'])].head(TOP_N - len(selected))
    selected = pd.concat([selected, remaining])
    print(f"Note: Only {len(trending)} above trend. Filled {len(remaining)} slots from below-trend.")

print("\n" + "=" * 90)
print(f"SELECTED PORTFOLIO — TOP {len(selected)} INSTRUMENTS")
print("=" * 90)
for i, (_, row) in enumerate(selected.iterrows(), 1):
    trend_icon = "ABOVE" if row['above_trend'] else "BELOW"
    print(f"  #{i:2d} | {row['name']:12s} | {row['asset_class']:10s} | Strategy: {row['best_strategy']:8s} | "
          f"Mom: {row['momentum_12_1']:+.1%} | Quality: {row['frog_score']:.3f} | Trend: {trend_icon} | "
          f"Vol: {row['realized_vol']:.1%}")

print("\n  PORTFOLIO ALLOCATION:")
print(f"  Equal weight: {1/len(selected):.1%} per instrument")
print(f"  With $100k FTMO: ${100000/len(selected):,.0f} per instrument")

# Strategy distribution
strat_dist = selected['best_strategy'].value_counts()
print(f"\n  STRATEGY MIX:")
for strat, count in strat_dist.items():
    print(f"    {strat}: {count} instruments ({count/len(selected):.0%})")

asset_dist = selected['asset_class'].value_counts()
print(f"\n  ASSET CLASS MIX:")
for ac, count in asset_dist.items():
    print(f"    {ac}: {count} instruments ({count/len(selected):.0%})")

## Historical Backtest: Quarterly Rebalanced QM Portfolio

The backtest below simulates what would have happened if we:
1. Every quarter, ranked all FTMO instruments by 12-1 momentum + quality
2. Selected the top 8 above their SMA(200)
3. Equal-weighted them
4. Held for 1 quarter, then rebalanced

This is a **time-series backtest** — at each rebalance date, we only use data available at that time.

In [ ]:
# HISTORICAL QUARTERLY REBALANCED BACKTEST

# Generate quarterly rebalance dates
all_dates = pd.DatetimeIndex(sorted(set.union(*[set(v.index) for v in all_data.values()])))
# Get quarter-end dates where we have at least 252 days of history
min_date = pd.Timestamp(START_DATE) + pd.DateOffset(months=13)  # Need 12 months + 1 month skip
quarterly_dates = pd.date_range(start=min_date, end=all_dates[-1], freq='QS')

print(f"Backtest period: {quarterly_dates[0].date()} to {all_dates[-1].date()}")
print(f"Rebalance dates: {len(quarterly_dates)} quarters")
print()

# Build a combined daily returns DataFrame for all instruments
returns_df = pd.DataFrame()
for ticker, close in all_data.items():
    returns_df[ticker] = close.pct_change()

portfolio_returns = []
holdings_history = []

for q_idx in range(len(quarterly_dates)):
    rebal_date = quarterly_dates[q_idx]
    next_rebal = quarterly_dates[q_idx + 1] if q_idx + 1 < len(quarterly_dates) else all_dates[-1]
    
    # Score instruments using only data before rebalance date
    scores = compute_momentum_scores(all_data, rebal_date)
    
    if len(scores) == 0:
        continue
    
    # Select: above trend first, then fill
    above = scores[scores['above_trend'] == True].head(TOP_N)
    if len(above) < TOP_N:
        below = scores[~scores['ticker'].isin(above['ticker'])].head(TOP_N - len(above))
        selection = pd.concat([above, below])
    else:
        selection = above
    
    selected_tickers = selection['ticker'].tolist()
    n_selected = len(selected_tickers)
    
    if n_selected == 0:
        continue
    
    # Equal weight returns for the holding period
    period_mask = (returns_df.index > rebal_date) & (returns_df.index <= next_rebal)
    period_returns = returns_df.loc[period_mask, selected_tickers]
    
    # Equal weight daily portfolio return
    daily_port_ret = period_returns.mean(axis=1)  # equal weight
    portfolio_returns.append(daily_port_ret)
    
    holdings_history.append({
        'rebal_date': rebal_date,
        'instruments': [FTMO_UNIVERSE[t]['name'] for t in selected_tickers if t in FTMO_UNIVERSE],
        'n_above_trend': len(above),
        'top_momentum': selection.iloc[0]['name'] if len(selection) > 0 else 'N/A'
    })
    
    names = ', '.join([FTMO_UNIVERSE[t]['name'] for t in selected_tickers if t in FTMO_UNIVERSE])
    print(f"  {rebal_date.date()} | {n_selected} instruments | {names}")

# Combine all periods
if portfolio_returns:
    port_daily = pd.concat(portfolio_returns).sort_index()
    port_daily = port_daily[~port_daily.index.duplicated(keep='first')]
    print(f"\nBacktest: {len(port_daily)} trading days")
    print(f"Date range: {port_daily.index[0].date()} to {port_daily.index[-1].date()}")
else:
    print("ERROR: No portfolio returns generated")

In [ ]:
# PERFORMANCE ANALYSIS

if len(port_daily) > 0:
    # Portfolio metrics
    cum_ret = (1 + port_daily).cumprod()
    total_ret = cum_ret.iloc[-1] - 1
    years = (port_daily.index[-1] - port_daily.index[0]).days / 365.25
    ann_ret = (1 + total_ret) ** (1/years) - 1
    ann_vol = port_daily.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    sortino_rets = port_daily[port_daily < 0]
    down_vol = sortino_rets.std() * np.sqrt(252) if len(sortino_rets) > 0 else 0
    sortino = ann_ret / down_vol if down_vol > 0 else 0
    peak = cum_ret.cummax()
    drawdown = (cum_ret - peak) / peak
    max_dd = drawdown.min()
    calmar = ann_ret / abs(max_dd) if abs(max_dd) > 0 else 0
    
    # SPY benchmark
    spy_close = all_data.get('^GSPC', None)
    if spy_close is not None:
        spy_rets = spy_close.pct_change().reindex(port_daily.index).fillna(0)
        spy_cum = (1 + spy_rets).cumprod()
        spy_total = spy_cum.iloc[-1] - 1
        spy_ann = (1 + spy_total) ** (1/years) - 1
        spy_vol = spy_rets.std() * np.sqrt(252)
        spy_sharpe = spy_ann / spy_vol if spy_vol > 0 else 0
        spy_dd = ((spy_cum - spy_cum.cummax()) / spy_cum.cummax()).min()
    
    print("=" * 70)
    print("QUANTITATIVE MOMENTUM FTMO PORTFOLIO — BACKTEST RESULTS")
    print("=" * 70)
    print(f"  {'Metric':25s} {'QM Portfolio':>15s} {'SPY B&H':>15s}")
    print("  " + "-" * 57)
    print(f"  {'Total Return':25s} {total_ret:>14.1%} {spy_total:>14.1%}")
    print(f"  {'Annualized Return':25s} {ann_ret:>14.1%} {spy_ann:>14.1%}")
    print(f"  {'Annualized Volatility':25s} {ann_vol:>14.1%} {spy_vol:>14.1%}")
    print(f"  {'Sharpe Ratio':25s} {sharpe:>14.3f} {spy_sharpe:>14.3f}")
    print(f"  {'Sortino Ratio':25s} {sortino:>14.3f} {'—':>15s}")
    print(f"  {'Max Drawdown':25s} {max_dd:>14.1%} {spy_dd:>14.1%}")
    print(f"  {'Calmar Ratio':25s} {calmar:>14.3f} {'—':>15s}")
    print(f"  {'Years':25s} {years:>14.1f} {years:>14.1f}")
    print("=" * 70)
    
    # Win rate
    win_days = (port_daily > 0).sum()
    total_days = len(port_daily)
    print(f"  Daily win rate: {win_days/total_days:.1%} ({win_days}/{total_days} days)")
    
    # Monthly returns
    monthly_rets = port_daily.resample('M').apply(lambda x: (1+x).prod() - 1)
    win_months = (monthly_rets > 0).sum()
    print(f"  Monthly win rate: {win_months/len(monthly_rets):.1%} ({win_months}/{len(monthly_rets)} months)")
    print(f"  Best month: {monthly_rets.max():.1%} | Worst month: {monthly_rets.min():.1%}")

In [ ]:
# EQUITY CURVE + DRAWDOWN

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [3, 1]})
fig.patch.set_facecolor('#FFFFFF')
fig.suptitle('Quantitative Momentum FTMO Portfolio', fontsize=16, fontweight='bold', color='#1A1D23')

# Equity
ax1.plot(cum_ret.index, cum_ret.values * 100000, color='#2ecc71', linewidth=2, label='QM Portfolio')
if spy_close is not None:
    ax1.plot(spy_cum.index, spy_cum.values * 100000, color='gray', linewidth=1, linestyle='--', alpha=0.6, label='SPY B&H')
ax1.set_facecolor('#FFFFFF'); ax1.tick_params(colors='#5A6270'); ax1.grid(True, alpha=0.1)
ax1.set_ylabel('Portfolio Value ($)', color='#5A6270', fontsize=12)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.legend(fontsize=10, facecolor='#F7F8FA', edgecolor='#E2E5EB', labelcolor='#1A1D23')
for spine in ax1.spines.values(): spine.set_color('#E2E5EB')

# Stats box
stats = f"Sharpe: {sharpe:.3f}  |  Return: {ann_ret:.1%}/yr  |  MaxDD: {max_dd:.1%}  |  Calmar: {calmar:.2f}"
ax1.text(0.5, 0.02, stats, transform=ax1.transAxes, fontsize=9, ha='center', color='#5A6270',
         family='monospace', bbox=dict(boxstyle='round,pad=0.4', facecolor='#F7F8FA', edgecolor='#E2E5EB'))

# Drawdown
ax2.fill_between(drawdown.index, drawdown.values * 100, 0, color='#e74c3c', alpha=0.5)
ax2.set_facecolor('#FFFFFF'); ax2.tick_params(colors='#5A6270'); ax2.grid(True, alpha=0.1)
ax2.set_ylabel('Drawdown %', color='#5A6270', fontsize=11)
for spine in ax2.spines.values(): spine.set_color('#E2E5EB')

plt.tight_layout(); plt.show()

In [ ]:
# QUARTERLY HOLDINGS HISTORY

print("=" * 90)
print("HOLDINGS HISTORY — What the QM model selected each quarter")
print("=" * 90)
for h in holdings_history:
    instruments = ', '.join(h['instruments'])
    print(f"  {h['rebal_date'].date()} | Above trend: {h['n_above_trend']} | Top: {h['top_momentum']:12s} | {instruments}")

# Frequency analysis: which instruments appear most often?
all_instruments = []
for h in holdings_history:
    all_instruments.extend(h['instruments'])

freq = pd.Series(all_instruments).value_counts()
print(f"\nMOST FREQUENTLY SELECTED (across {len(holdings_history)} quarters):")
for name, count in freq.head(15).items():
    pct = count / len(holdings_history) * 100
    bar = '#' * int(pct / 3)
    print(f"  {name:12s}  {count:3d}x ({pct:5.1f}%)  {bar}")

In [ ]:
# FTMO MONTE CARLO SIMULATION

dr = port_daily.values.ravel()
dr = dr[~np.isnan(dr)]

N_SIM = 10000; DAYS = 30; ACCOUNT = 100000
np.random.seed(42)
n_passed = n_blown_t = n_blown_d = 0
final_eqs = np.zeros(N_SIM)
sample_paths = []

for s in range(N_SIM):
    eq = ACCOUNT; path = [ACCOUNT]
    sim_rets = np.random.choice(dr, size=DAYS, replace=True)
    blown = False
    for d in range(DAYS):
        day_start = eq
        eq *= (1 + sim_rets[d])
        path.append(eq)
        if (eq - day_start) / ACCOUNT < -0.05:
            n_blown_d += 1; blown = True; break
        if (eq - ACCOUNT) / ACCOUNT < -0.10:
            n_blown_t += 1; blown = True; break
        if (eq - ACCOUNT) / ACCOUNT >= 0.10:
            n_passed += 1; blown = True; break
    while len(path) < DAYS + 1:
        path.append(path[-1])
    final_eqs[s] = path[-1]
    if s < 200:
        sample_paths.append(path)

n_still = N_SIM - n_passed - n_blown_t - n_blown_d
pass_rate = n_passed / N_SIM * 100
verdict = "FAVORABLE" if pass_rate >= 50 else "POSSIBLE" if pass_rate >= 25 else "CHALLENGING" if pass_rate >= 10 else "UNLIKELY"

print("=" * 70)
print(f"FTMO CHALLENGE MONTE CARLO — {N_SIM:,} SIMULATIONS")
print("=" * 70)
print(f"  Pass Rate:      {pass_rate:.1f}% ({n_passed:,} passed)")
print(f"  Blown (total):  {n_blown_t:,}")
print(f"  Blown (daily):  {n_blown_d:,}")
print(f"  Still trading:  {n_still:,}")
print(f"  Median equity:  ${np.median(final_eqs):,.0f}")
print(f"  Verdict:        {verdict}")
print("=" * 70)

# Plot
fig, ax = plt.subplots(figsize=(14, 8))
fig.patch.set_facecolor('#FFFFFF'); ax.set_facecolor('#FFFFFF')
for spine in ax.spines.values(): spine.set_color('#E2E5EB')
ax.tick_params(colors='#5A6270'); ax.grid(True, alpha=0.15, color='#E2E5EB')

for path in sample_paths:
    c = '#2ca02c' if path[-1] >= 110000 else ('#e74c3c' if path[-1] <= 90000 else '#8C95A3')
    ax.plot(range(DAYS+1), path, color=c, alpha=0.2, linewidth=0.5)
ax.axhline(110000, color='#2ca02c', linestyle='--', linewidth=2.5, label='10% Target ($110k)')
ax.axhline(90000, color='#e74c3c', linestyle='--', linewidth=2.5, label='10% Max Loss ($90k)')
ax.axhline(100000, color='#1A1D23', linestyle='-', linewidth=0.5, alpha=0.3)

ax.set_title(f'FTMO Monte Carlo — QM Portfolio — {pass_rate:.1f}% Pass Rate ({verdict})', fontsize=14, fontweight='bold', color='#1A1D23')
ax.set_xlabel('Trading Day', color='#5A6270'); ax.set_ylabel('Equity ($)', color='#5A6270')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=10, facecolor='#F7F8FA', edgecolor='#E2E5EB', labelcolor='#1A1D23')
plt.tight_layout(); plt.show()

In [ ]:
# CURRENT TRADING RECOMMENDATIONS

print("=" * 70)
print("CURRENT QM TRADING RECOMMENDATIONS")
print("=" * 70)
print(f"Date: {pd.Timestamp.now().date()}")
print(f"Method: 12-1 Momentum + Frog-in-the-Pan Quality")
print(f"Trend Filter: SMA({TREND_SMA})")
print()

for i, (_, row) in enumerate(selected.iterrows(), 1):
    trend_status = "ABOVE SMA" if row['above_trend'] else "BELOW SMA"
    print(f"  #{i} {row['name']:12s} ({row['asset_class']:10s})")
    print(f"     Strategy: {row['best_strategy']}")
    print(f"     12-1 Momentum: {row['momentum_12_1']:+.1%}")
    print(f"     Quality Score: {row['frog_score']:.3f} (higher = smoother momentum)")
    print(f"     Trend: {trend_status}")
    print(f"     Realized Vol: {row['realized_vol']:.1%}")
    print()

print("NEXT STEPS:")
print(f"  1. Run each selected instrument through its recommended strategy notebook")
print(f"  2. Use the regime filter to confirm timing")
print(f"  3. Equal-weight across all {len(selected)} instruments")
print(f"  4. Rebalance next quarter")
print(f"\nQuick backtest commands:")
for _, row in selected.iterrows():
    print(f"  !python scripts/quick_backtest.py --strategy {row['best_strategy']} --ticker {row['ticker']}")

In [ ]:
# CORRELATION MATRIX OF SELECTED PORTFOLIO

selected_tickers = selected['ticker'].tolist()
selected_names = [FTMO_UNIVERSE[t]['name'] for t in selected_tickers]

# Get overlapping returns
corr_rets = returns_df[selected_tickers].dropna()
corr_matrix = corr_rets.corr()

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#FFFFFF')
ax.set_facecolor('#FFFFFF')

im = ax.imshow(corr_matrix.values, cmap='RdYlGn_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(selected_names))); ax.set_xticklabels(selected_names, rotation=45, ha='right', fontsize=10, color='#5A6270')
ax.set_yticks(range(len(selected_names))); ax.set_yticklabels(selected_names, fontsize=10, color='#5A6270')

for i in range(len(selected_names)):
    for j in range(len(selected_names)):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.5 else '#5A6270'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Portfolio Correlation Matrix', fontsize=14, fontweight='bold', color='#1A1D23')
for spine in ax.spines.values(): spine.set_color('#E2E5EB')
plt.tight_layout(); plt.show()

# Flag high correlations
print("\nCORRELATION WARNINGS (> 0.60):")
warned = False
for i in range(len(selected_tickers)):
    for j in range(i+1, len(selected_tickers)):
        corr = corr_matrix.values[i, j]
        if abs(corr) > 0.60:
            print(f"  {selected_names[i]} <-> {selected_names[j]}: {corr:.3f}")
            warned = True
if not warned:
    print("  None — good diversification!")

avg_corr = corr_matrix.values[np.triu_indices_from(corr_matrix.values, k=1)].mean()
print(f"\n  Average pairwise correlation: {avg_corr:.3f}")
print(f"  {'Good diversification' if avg_corr < 0.3 else 'Moderate overlap' if avg_corr < 0.5 else 'High overlap — consider reducing'}")

In [ ]:
# SAVE RESULTS
import json, os, datetime

EXPORT_DIR = "strategy_exports"
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    EXPORT_DIR = "/content/drive/MyDrive/strategy_exports"
except:
    pass

out_dir = os.path.join(EXPORT_DIR, "Quantitative_Momentum", "FTMO_Portfolio")
os.makedirs(out_dir, exist_ok=True)

run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results = {
    "metadata": {
        "run_id": run_id,
        "strategy": "Quantitative_Momentum_FTMO",
        "method": "12-1 Momentum + Frog-in-the-Pan Quality (Alpha Architect)",
        "universe_size": len(FTMO_UNIVERSE),
        "instruments_loaded": len(all_data),
        "top_n": TOP_N,
        "rebalance_freq": REBALANCE_FREQ,
        "trend_filter": f"SMA({TREND_SMA})",
        "start_date": START_DATE,
    },
    "backtest_metrics": {
        "total_return": float(total_ret),
        "annualized_return": float(ann_ret),
        "annualized_volatility": float(ann_vol),
        "sharpe_ratio": float(sharpe),
        "sortino_ratio": float(sortino),
        "max_drawdown": float(max_dd),
        "calmar_ratio": float(calmar),
        "years": float(years),
    },
    "ftmo_monte_carlo": {
        "pass_rate": float(pass_rate),
        "n_simulations": N_SIM,
        "n_passed": int(n_passed),
        "verdict": verdict,
    },
    "current_portfolio": selected[['name', 'asset_class', 'best_strategy', 'momentum_12_1', 'frog_score', 'above_trend']].to_dict('records'),
    "holdings_history": [{
        'date': str(h['rebal_date'].date()),
        'instruments': h['instruments']
    } for h in holdings_history],
}

with open(os.path.join(out_dir, "qm_summary.json"), 'w') as f:
    json.dump(results, f, indent=2, default=str)

port_daily.to_csv(os.path.join(out_dir, "qm_daily_returns.csv"))

print(f"Saved to {out_dir}/")
print(f"  qm_summary.json")
print(f"  qm_daily_returns.csv")
print(f"\nRun ID: {run_id}")

# ════════════════════════════════════════════════════════════════
# LOG TO GOOGLE SHEETS
# ════════════════════════════════════════════════════════════════
try:
    from lib.sheets_logger import SheetsLogger
    _logger = SheetsLogger()
    # Format QM results into summary.json-compatible format
    _logger.log_portfolio(results)
except Exception as _e:
    print(f"⚠️ Sheets logging skipped: {_e}")
